In [0]:
%load_ext autoreload
%autoreload 2
# Enables autoreload; learn more at https://docs.databricks.com/en/files/workspace-modules.html#autoreload-for-python-modules
# To disable autoreload; run %autoreload 0

In [0]:
%run ../../config/metadata_setup

In [0]:
from validators.data_validator import DataValidator

In [0]:
from pyspark.sql.types import IntegerType, StringType, StructField, StructType, DecimalType
from pyspark.sql.functions import col, lit, filter,current_timestamp
from datetime import datetime

import os

In [0]:
class OrderBronzeStreaming:
    def __init__(self, spark):
        self.spark = spark
        self.catalog_name = CONFIG['catalog']['name']
        self.schema_name = CONFIG['catalog']['schema']

        # Paths
        self.source_path = CONFIG['paths']['raw_data']
        self.schema_path = CONFIG['paths']['schemas']
        self.checkpoint_path = CONFIG['paths']['checkpoints']

        # sub paths
        self.sub_path = CONFIG['subpaths']['orders']

        # table name
        self.table_name = TABLES['bronze']['orders']

        # data validation table
        self.validation_table = TABLES['data_validation']['vali_table']

        # Schema
        self.schema = self._define_schema()

    # -------------------------------
    # Define schema
    # -------------------------------
    def _define_schema(self):
        return StructType([
            StructField('order_id', StringType(), True),
            StructField('order_date', StringType(), True),
            StructField('customer_name', StringType(), True),
            StructField('state', StringType(), True),
            StructField('city', StringType(), True)
        ])

    # --------------------------------
    # Read streaming data
    # --------------------------------
    def read_stream(self):
        return (
            self.spark.readStream
            .format('cloudFiles')
            .option('cloudFiles.format', 'csv')
            .option('cloudFiles.schemaLocation', f"{self.schema_path}/_{self.table_name}")
            .option('cloudFiles.schemaEvolutionMode', 'rescue')
            .option("delimiter", ",")
            .schema(self.schema)
            .load(f"{self.source_path}/{self.sub_path}")           
        )

    # --------------------------------
    # header cleaning
    # ---------------------------------
    def clean_header(self, df, header_col:str, header_value:str):
        """
        Removing header row
        """ 
        try:
            return df.filter(col(header_col) != header_value)
        except Exception as e:
            print(f'Error: clean_header: {e}')

    # ----------------------------------
    # add file metadata
    # ----------------------------------
    def add_metadata(self, df):
        return (df.withColumn('ingest_ts', lit(datetime.now()))
                  .withColumn('file_name', col("_metadata.file_name"))
                  .withColumn('file_path', col("_metadata.file_path"))
                  .withColumn('file_modification_time', col("_metadata.file_modification_time"))
                  .withColumn('file_size', col("_metadata.file_size"))
                  .drop("_metadata"))

    # ---------------------------------
    # display table
    # ---------------------------------
    def display_tbl(self):
        return self.spark.sql(f"SELECT * FROM {self.catalog_name}.{self.schema_name}.{self.table_name}")

    # --------------------------------
    # Only for droping table
    #---------------------------------
    #def drop_table(self):
    #    dbutils.fs.rm(f"{self.schema_path}/_{self.table_name}", True)
    #    dbutils.fs.rm(f"{self.checkpoint_path}/_{self.table_name}", True)
    #    self.spark.sql(f"DROP TABLE IF EXISTS {self.catalog_name}.{self.schema_name}.{self.table_name}")
    
    # -----------------------------------
    # create validation datafreame
    # ------------------------------------
    def validation_df(self, df):
        validator = DataValidator(df, self.table_name)
        results = []
        results.extend(validator.check_not_null(["order_id", "order_date"]))
        results.append(validator.check_duplicate(["order_id"]))
        results.append(validator.check_row_count())
        return results

    def _process_batch(self, batch_df, batch_id):
        # Write to bronze table
        (
            batch_df.write
            .format("delta")
            .mode("append")
            .saveAsTable(f"{self.catalog_name}.{self.schema_name}.{self.table_name}")
        )

        # Run validations (NOW SAFE – batch_df is NOT streaming)
        validation_results = self.validation_df(batch_df)

        # Example: persist validation results
        if validation_results:
            validation_df = (
                self.spark.createDataFrame(validation_results)
                .withColumn("validation_timestamp", current_timestamp())
            )

        validation_df.write.mode("append").saveAsTable(f"{self.catalog_name}.{self.schema_name}.{self.validation_table}")

    # ---------------------------------
    # write to delta table (streaming)
    # ---------------------------------
    def write_stream(self, df):
        try:
            query = (
                df.writeStream
                .format('delta')
                .option('checkpointLocation', f"{self.checkpoint_path}/_{self.table_name}")
                .trigger(availableNow=True)
                .foreachBatch(self._process_batch)
                .start()
            )   
            query.awaitTermination()
        except Exception as e:
            print(f'Error: write_stream: {e}')  

    def run(self):
        df = self.read_stream()
        df = self.add_metadata(df)
        df = self.clean_header(df, 'order_id', 'Order ID')
        self.write_stream(df)           

In [0]:
obj = OrderBronzeStreaming(spark)
obj.run()

In [0]:
#obj.drop_table()